In [ ]:
# Step 1: Install Required Libraries
!pip install -q transformers accelerate datasets peft bitsandbytes torch pandas numpy scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [ ]:
# Step 2: Import Libraries
import torch
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import json
from google.colab import files
import os
import shutil

In [ ]:
# Step 3: Load Tokenizer
model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using Device: {device}")

# Step 4: Load the Modified Bengali Review Dataset
df = pd.read_csv("Modified_Bengali_Review_Dataset.csv")  # Replace with actual dataset path

def preprocess_text(text):
    return text if isinstance(text, str) else ""

df["Reviews"] = df["Reviews"].apply(preprocess_text)  # Ensure all text is valid strings

df.rename(columns={"Sentiment": "labels"}, inplace=True)  # Ensure correct column naming

df["labels"] = df["labels"].astype(int)  # Ensure labels are integers

# Convert dataset to Hugging Face Dataset format
dataset = Dataset.from_pandas(df)

def tokenize_function(examples):
    return tokenizer(examples["Reviews"], padding="max_length", truncation=True, max_length=256)

dataset = dataset.map(tokenize_function, batched=True)

# Split into Train and Test
dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = dataset["train"]
test_dataset = dataset["test"]

print("✅ Modified Bengali Review Dataset loaded and processed!")

✅ Using Device: cuda


Map:   0%|          | 0/11807 [00:00<?, ? examples/s]

✅ Modified Bengali Review Dataset loaded and processed!


In [ ]:
# Step 5: Load Model
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, num_labels=len(set(df["labels"]))
).to(device)

print("✅ mBERT Model Loaded Successfully!")

# Step 6: Configure LoRA for Fine-Tuning
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    inference_mode=False,
    r=8,
    lora_alpha=32,
    lora_dropout=0.1
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print("✅ LoRA Applied Successfully!")

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ mBERT Model Loaded Successfully!
trainable params: 296,450 || all params: 178,151,428 || trainable%: 0.1664
✅ LoRA Applied Successfully!


In [ ]:
# Step 7: Define Training Arguments
training_args = TrainingArguments(
    output_dir="./mb-bert_results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    fp16=torch.cuda.is_available(),
    report_to="none",
    push_to_hub=False
)

print("✅ Training Arguments Set!")


✅ Training Arguments Set!


/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
# Step 8: Define Evaluation Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="weighted")
    return {"accuracy": acc, "precision": precision, "recall": recall, "f1-score": f1}

print("✅ Evaluation Metrics Defined!")

# Step 9: Fine-Tune the Model
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

print("🚀 Training mBERT with LoRA Started...")
trainer.train()
print("✅ Training Completed!")


✅ Evaluation Metrics Defined!
🚀 Training mBERT with LoRA Started...


<ipython-input-10-36590740cfb3>:12: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1-score
1,0.394600,0.317026,0.861558,0.860732,0.861558,0.861114
2,0.273600,0.323691,0.879340,0.876600,0.879340,0.876393
3,0.248100,0.287606,0.890347,0.894832,0.890347,0.891875
4,0.401600,0.308946,0.897968,0.896402,0.897968,0.896784
5,0.180800,0.295342,0.902202,0.901886,0.902202,0.902034


✅ Training Completed!


In [ ]:
# Step 10: Evaluate the Model
eval_results = trainer.evaluate()
print("🔍 Evaluating the Model...")
for key, value in eval_results.items():
    print(f"{key}: {value}")

🔍 Evaluating the Model...
eval_loss: 0.29534152150154114
eval_accuracy: 0.9022015241320914
eval_precision: 0.9018858334161306
eval_recall: 0.9022015241320914
eval_f1-score: 0.9020342070653433
eval_runtime: 10.4152
eval_samples_per_second: 226.784
eval_steps_per_second: 28.42
epoch: 5.0


In [ ]:
# Step 11: Save the Fine-Tuned Model
trainer.save_model("./mb_bert_fine_tuned")
tokenizer.save_pretrained("./mb_bert_fine_tuned")

# Save Evaluation Results
with open("mb_bert_results.json", "w") as f:
    json.dump(eval_results, f)

# Zip Model Folder
shutil.make_archive("mb_bert_fine_tuned", 'zip', "./mb_bert_fine_tuned")

# Download Model & Results
files.download("mb_bert_fine_tuned.zip")
files.download("mb_bert_results.json")

print("✅ Fine-Tuned mBERT Model & Results Saved Successfully!")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Fine-Tuned mBERT Model & Results Saved Successfully!
